# Results Interpretation

This notebook summarizes the results obtained from the Isolation Forest anomaly detection model and the SHAP explainability analysis on the HDFS log dataset.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../datasets/HDFS_2k_structured.csv")

print("Dataset Shape:", df.shape)

Dataset Shape: (2000, 6)


In [3]:
df["Message_Length"] = df["Message"].str.len()

df["Has_Block_ID"] = df["Message"].str.contains("blk_", regex=False).astype(int)

df["Has_PacketResponder"] = df["Message"].str.contains("PacketResponder", regex=False).astype(int)

df["Has_Error"] = df["Message"].str.contains("error", case=False, regex=False).astype(int)

df["Is_WARN"] = (df["Level"] == "WARN").astype(int)

In [4]:
from sklearn.ensemble import IsolationForest

features = [
    "Message_Length",
    "Has_Block_ID",
    "Has_PacketResponder",
    "Has_Error",
    "Is_WARN"
]

X = df[features]

model = IsolationForest(
    contamination=0.05,
    random_state=42
)

model.fit(X)

df["Anomaly"] = model.predict(X)

In [5]:
print("Total Logs:", len(df))

print("Detected Anomalies:", (df["Anomaly"] == -1).sum())

print("Normal Logs:", (df["Anomaly"] == 1).sum())

Total Logs: 2000
Detected Anomalies: 88
Normal Logs: 1912


## Dataset Summary

The Isolation Forest model analysed a total of 2,000 HDFS log entries.

Among these logs, 88 entries were identified as anomalous, while the remaining 1,912 entries were classified as normal behaviour.

This indicates that approximately 4–5% of the dataset contained unusual log patterns requiring further investigation.

In [6]:
component_counts = (
    df[df["Anomaly"] == -1]["Component"]
    .value_counts()
)

print(component_counts)

Component
dfs.DataNode$DataXceiver:        35
dfs.DataNode$PacketResponder:    30
dfs.DataBlockScanner:            20
dfs.FSNamesystem:                 3
Name: count, dtype: int64


## Component Analysis

The anomalous log entries were concentrated within a small number of HDFS components.

The **dfs.DataNode$DataXceiver** component produced the highest number of detected anomalies, indicating that data transmission and packet communication activities contributed significantly to unusual system behaviour.

Other components generated comparatively fewer anomalous log entries.

In [7]:
level_counts = (
    df[df["Anomaly"] == -1]["Level"]
    .value_counts()
)

print(level_counts)

Level
INFO    53
WARN    35
Name: count, dtype: int64


## Log Level Analysis

Most anomalous log entries belonged to the **INFO** logging level.

Although warning messages were present, the Isolation Forest model demonstrated that unusual behaviour can occur even within normal informational logs.

This highlights the importance of analysing log content rather than relying solely on log severity.

In [8]:
avg_normal = df[df["Anomaly"] == 1]["Message_Length"].mean()

avg_anomaly = df[df["Anomaly"] == -1]["Message_Length"].mean()

print("Average normal message length:", avg_normal)
print("Average anomaly message length:", avg_anomaly)

Average normal message length: 93.95135983263599
Average anomaly message length: 127.42045454545455


## Message Length Analysis

The average length of anomalous log messages was greater than that of normal log messages.

This observation supports the SHAP analysis, which identified **Message_Length** as the most influential feature in the Isolation Forest model.

Longer log messages generally contained more operational details, making them more likely to be recognised as anomalous.

## SHAP Explainability Analysis

SHAP (SHapley Additive exPlanations) was used to interpret the predictions generated by the Isolation Forest model.

The SHAP summary plot showed that **Message_Length** was the most influential feature in anomaly detection. Longer log messages generally contributed more strongly towards anomaly predictions.

The **Has_PacketResponder** feature was identified as the second most important contributor, indicating that packet communication activities played an important role in distinguishing anomalous log behaviour.

The **Is_WARN** feature also contributed to anomaly detection, although its influence was smaller compared to message length and packet responder information.

The remaining features, **Has_Error** and **Has_Block_ID**, contributed relatively less to the model's decisions.

Overall, the SHAP analysis demonstrates that the Isolation Forest model primarily relied on message characteristics and communication-related features when identifying abnormal HDFS log events.

## Key Findings

- A total of **2,000** HDFS log entries were analysed.
- The Isolation Forest model detected **88 anomalous** log entries.
- The **dfs.DataNode$DataXceiver** component generated the highest number of anomalous events.
- Anomalous log messages were longer on average than normal log messages.
- SHAP explainability identified **Message_Length** as the most influential feature, followed by **Has_PacketResponder** and **Is_WARN**.
- The explainability analysis improved the transparency of the anomaly detection model by identifying which features contributed most to each prediction.

## Conclusion

The Isolation Forest model successfully detected anomalous HDFS log entries using engineered textual features.

The integration of SHAP explainability provided valuable insights into the model's decision-making process, demonstrating that message length and packet communication features were the strongest indicators of anomalous system behaviour.

These findings support the objective of developing an explainable machine learning framework for system log anomaly detection.